# 14 - 模型评估 (AI Infra 视角)

本节从 **工程实现** 角度理解 LLM 评估体系：
- Loss vs BPB (Bits Per Byte)
- CORE 评估：多选题 / Schema / 语言建模
- 下游任务：MMLU / ARC / GSM8K / HumanEval
- 基座模型 vs 聊天模型的评估差异
- 评估的工程实现

> 参考 nanochat/nanochat/loss_eval.py, core_eval.py, tasks/

In [ ]:
import torch
import torch.nn.functional as F
import math

## 1. 核心概念 (30秒版)

```
评估分两个层面:

① 训练质量 (连续指标)
   Loss / BPB → 模型预测下一个 token 有多准
   越低越好, 但无法直接反映 "模型有多聪明"

② 下游能力 (离散指标)
   MMLU → 知识面 (多学科选择题)
   ARC  → 科学推理 (小学/初中科学题)
   GSM8K → 数学推理 (小学应用题)
   HumanEval → 编程能力 (写 Python 函数)
   
   这些才是用户真正关心的 "模型有多聪明"
```

## 2. Loss vs BPB

### 为什么不直接用 Loss？

Loss (cross-entropy, nats) 依赖于词表大小和分词方式：

```
同一段文本 "Hello World":
  词表 A (50k): 2 个 token → loss 在 2 个 token 上求平均
  词表 B (100k): 3 个 token → loss 在 3 个 token 上求平均
  
两者的 loss 无法直接比较!
```

### BPB (Bits Per Byte) 解决这个问题

```
BPB = total_nats / (ln(2) × total_bytes)

不管你用什么分词器, "Hello World" 的字节数是固定的 (11 bytes)
所以 BPB 可以跨模型、跨词表公平比较
```

In [ ]:
# BPB 计算演示

def compute_bpb_simple(losses, token_byte_lengths):
    """
    losses: 每个 token 的 cross-entropy loss (nats)
    token_byte_lengths: 每个 token 对应多少字节
    """
    # 只计算有效 token (字节数 > 0 的, 排除 special tokens)
    valid = token_byte_lengths > 0
    total_nats = (losses * valid.float()).sum().item()
    total_bytes = token_byte_lengths[valid].sum().item()
    bpb = total_nats / (math.log(2) * total_bytes)
    return bpb

# 模拟: 5 个 token, 每个 token 有不同的字节长度
losses = torch.tensor([3.0, 2.5, 4.0, 1.5, 3.2])  # nats
token_bytes = torch.tensor([0, 3, 5, 2, 4])  # 第一个是 BOS, 0 字节 → 不计入

bpb = compute_bpb_simple(losses, token_bytes)
avg_loss = losses[1:].mean().item()  # 排除 BOS 的平均 loss

print(f"平均 Loss: {avg_loss:.4f} nats")
print(f"BPB: {bpb:.4f} bits/byte")
print()
print("BPB 的直觉: 模型平均需要多少 bit 来编码原始文本的每个字节")
print("  BPB ≈ 1.0 → 非常好 (接近最优压缩)")
print("  BPB ≈ 2.0 → 一般")
print("  BPB ≈ 4.0 → 较差 (模型没学到多少)")

In [ ]:
# nanochat evaluate_bpb 的核心逻辑 (简化版)

def evaluate_bpb_simplified(model, batches, steps, token_bytes):
    """
    nanochat/loss_eval.py 的简化版
    
    关键设计:
    1. loss_reduction='none' → 得到每个 token 的 loss (不是平均)
    2. token_bytes[y] → 查表得到每个目标 token 的字节数
    3. special tokens 的字节数为 0 → 自动排除
    4. 分布式: all_reduce(SUM) 汇总所有 rank 的 nats 和 bytes
    """
    total_nats = 0.0
    total_bytes = 0
    
    for x, y in batches:
        # 不做 reduction, 得到每个 token 的 loss
        loss2d = model(x, y, loss_reduction='none')  # (B, T)
        
        # 查表: 每个目标 token 有多少字节
        num_bytes2d = token_bytes[y.view(-1)]  # special tokens → 0
        
        # 累加 (字节为 0 的 token 不计入 nats)
        total_nats += (loss2d.view(-1) * (num_bytes2d > 0)).sum()
        total_bytes += num_bytes2d.sum()
    
    # 分布式汇总 (如果有多 GPU)
    # dist.all_reduce(total_nats, op=SUM)
    # dist.all_reduce(total_bytes, op=SUM)
    
    bpb = total_nats / (math.log(2) * total_bytes)
    return bpb

print("evaluate_bpb 的设计要点:")
print("  1. loss_reduction='none' 而不是 'mean'")
print("     → 需要逐 token 的 loss 来配合字节长度加权")
print("  2. token_bytes 是预计算的查找表")
print("     → 每个 token ID 对应多少字节, special tokens 为 0")
print("  3. 分布式 all_reduce(SUM)")
print("     → 先各自累加, 最后汇总, 比 all_reduce(AVG) 更精确")

## 3. CORE 评估 — 基座模型的能力测试

CORE (DCLM 论文) 是一套针对**基座模型**的评估体系。

基座模型没有经过 SFT，不会"回答问题"，只会"预测下一个 token"。
所以评估方式和聊天模型完全不同：

```
聊天模型评估:
  问: "法国的首都是哪里?"
  答: "巴黎"
  → 判断答案是否正确

基座模型评估:
  给模型看: "The capital of France is Paris"  vs  "The capital of France is London"
  → 哪个选项的 loss 更低 (困惑度更低)?
  → loss 更低的那个 = 模型认为更合理的答案
```

In [ ]:
# 基座模型评估的核心思路: 比较不同选项的 loss

# 模拟: 一道多选题
question = "The capital of France is"
choices = ["Paris", "London", "Berlin", "Madrid"]
correct_answer = 0  # Paris

# 模拟模型对每个选项的 loss
# (真实场景中是把 question+choice 拼接后过模型, 算 choice 部分的 loss)
torch.manual_seed(42)
simulated_losses = [1.2, 3.5, 4.1, 3.8]  # Paris 的 loss 最低

print(f"问题: {question}")
print()
for i, (choice, loss) in enumerate(zip(choices, simulated_losses)):
    marker = " ← 模型选择 (loss 最低)" if loss == min(simulated_losses) else ""
    correct = " ✓" if i == correct_answer else ""
    print(f"  {choice}: loss = {loss:.2f}{marker}{correct}")

predicted = simulated_losses.index(min(simulated_losses))
print(f"\n模型预测: {choices[predicted]} ({'正确' if predicted == correct_answer else '错误'})")

## 4. 三种评估任务类型

nanochat 的 `core_eval.py` 支持三种任务类型：

### Multiple Choice (多选题)
```
题目 (context) 相同, 选项 (continuation) 不同
→ 找共同前缀, 只比较选项部分的 loss

"The capital of France is [Paris]"    ← 比较这部分的 loss
"The capital of France is [London]"   ← 比较这部分的 loss
```

### Schema (模式题)
```
题目 (context) 不同, 但后续 (continuation) 相同
→ 找共同后缀, 只比较后缀部分的 loss

"[The cat sat on the mat,] it was comfortable"
"[The dog flew to the moon,] it was comfortable"
                             ↑ 比较这部分的 loss
```

### Language Modeling (语言建模)
```
给一段上下文, 预测续写
→ 逐 token 判断 argmax 是否和正确续写一致

context: "2 + 2 ="
continuation: " 4"
→ 模型在 "=" 之后预测的 argmax 是 " 4" 吗?
```

In [ ]:
# 多选题评估的实现细节

def find_common_prefix_length(sequences):
    """找到多个 token 序列的共同前缀长度"""
    min_len = min(len(s) for s in sequences)
    for i in range(min_len):
        if not all(s[i] == sequences[0][i] for s in sequences):
            return i
    return min_len

# 模拟 tokenize 后的多选题
# 题干: "The capital of France is" → 共同前缀
# 选项: 不同后缀
option_tokens = [
    [1, 50, 100, 200, 300, 77],       # "The capital of France is Paris"
    [1, 50, 100, 200, 300, 88, 99],   # "The capital of France is London"
    [1, 50, 100, 200, 300, 66],       # "The capital of France is Berlin"
]

prefix_len = find_common_prefix_length(option_tokens)
print(f"共同前缀长度: {prefix_len} tokens")
print()
for i, tokens in enumerate(option_tokens):
    prefix = tokens[:prefix_len]
    suffix = tokens[prefix_len:]
    print(f"  选项 {i}: 前缀 {prefix} + 选项 {suffix}")

print()
print("评估时只计算选项部分 (suffix) 的平均 loss")
print("loss 最低的选项 = 模型的答案")

## 5. 下游评估任务详解

| 任务 | 类型 | 测什么 | 题目数 | 评估方式 |
|------|------|--------|--------|----------|
| **MMLU** | 多选 | 多学科知识 (57 科目) | ~14,000 | 选项 loss 比较 |
| **ARC-Easy** | 多选 | 小学科学题 | ~2,300 | 选项 loss 比较 |
| **ARC-Challenge** | 多选 | 较难科学题 | ~1,100 | 选项 loss 比较 |
| **GSM8K** | 生成 | 小学数学应用题 | ~1,300 | 生成答案判对错 |
| **HumanEval** | 生成 | Python 编程 | 164 | 运行代码判对错 |

In [ ]:
# nanochat 的 Task 抽象

print("nanochat 的 Task 基类设计:")
print()
print("class Task:")
print("    eval_type     → 'categorical' 或 'generative'")
print("    num_examples() → 数据集大小")
print("    get_example(i) → 返回一个 conversation dict")
print("    evaluate(conv, response) → 判断是否正确")
print()
print("每个 Task 返回的 conversation 格式:")
print('{"messages": [')
print('    {"role": "user", "content": "Multiple Choice question: ..."},')
print('    {"role": "assistant", "content": "A"}')
print(']}')
print()

# ARC 的多选题格式示例
print("=== ARC 多选题示例 ===")
print()
question = "What is the boiling point of water?"
letters = ["A", "B", "C", "D"]
choices = ["50°C", "100°C", "150°C", "200°C"]

# nanochat 的 render_mc 格式
query = f"Multiple Choice question: {question}\n"
query += "".join([f"- {choice}={letter}\n" for letter, choice in zip(letters, choices)])
query += "\nRespond only with the letter of the correct answer."
print(query)
print()
print("注意: 选项格式是 'choice=letter' 而不是 'letter. choice'")
print("这样做的原因: 小模型对 token 边界敏感,")
print("把字母放在末尾能让模型更好地学会直接输出字母")

## 6. 基座模型 vs 聊天模型的评估

```
基座模型 (pretrain/base):
  不会回答问题, 只会续写文本
  评估方式: 比较不同选项的 loss (CORE 方法)
  代码: core_eval.py + loss_eval.py
  指标: BPB, CORE accuracy

聊天模型 (SFT/RL):
  能理解指令, 生成回答
  评估方式: 让模型生成答案, 判断对错
  代码: chat_eval.py + tasks/
  指标: 各任务的 accuracy, pass@k (HumanEval)
```

| | 基座模型评估 | 聊天模型评估 |
|--|------------|------------|
| 输入 | 完整文本 (含答案) | 问题 (不含答案) |
| 模型动作 | 计算 loss | 生成回答 |
| 判断方式 | 哪个选项 loss 低 | 回答是否正确 |
| 适用任务 | 选择题为主 | 选择题 + 开放题 |

In [ ]:
# 演示: forward_model 的实现
# 这是 core_eval.py 中评估的核心函数

def forward_model_demo(logits, input_ids):
    """
    给定模型输出的 logits 和输入 token ids,
    计算每个位置的自回归 loss 和 argmax 预测
    """
    B, T, V = logits.shape
    
    # 目标是下一个 token (左移一位)
    target_ids = torch.roll(input_ids, shifts=-1, dims=1)
    
    # 计算每个位置的 cross-entropy loss
    losses = F.cross_entropy(
        logits.view(B * T, V),
        target_ids.view(B * T),
        reduction='none'
    ).view(B, T)
    
    # 最后一个位置没有目标, 设为 nan
    losses[:, -1] = float('nan')
    
    # argmax 预测
    predictions = logits.argmax(dim=-1)
    
    return losses, predictions

# 模拟一个 batch
B, T, V = 2, 6, 100  # 2 个选项, 6 个 token, 100 词表
torch.manual_seed(0)
logits = torch.randn(B, T, V)
input_ids = torch.randint(0, V, (B, T))

losses, predictions = forward_model_demo(logits, input_ids)
print(f"每个位置的 loss (shape {losses.shape}):")
print(f"  选项 0: {[f'{l:.2f}' if not math.isnan(l) else 'nan' for l in losses[0].tolist()]}")
print(f"  选项 1: {[f'{l:.2f}' if not math.isnan(l) else 'nan' for l in losses[1].tolist()]}")
print()
print("多选题评估: 取选项部分 (去掉共同前缀) 的平均 loss, 最低者胜")

## 7. Few-Shot 评估

基座模型不理解指令, 需要通过**示例**引导：

```
0-shot (无示例):
  "Multiple Choice question: What is H2O?
   - Water=A
   - Fire=B"
  → 模型可能不知道该选字母

5-shot (给 5 个示例):
  "Multiple Choice question: What color is the sky? ...
   Answer: B
   
   Multiple Choice question: What is 2+2? ...
   Answer: A
   
   ... (3 more examples) ...
   
   Multiple Choice question: What is H2O?
   - Water=A
   - Fire=B"
  → 模型从示例中学会了格式, 表现更好
```

nanochat 用确定性随机采样保证评估可复现：
```python
rng = random.Random(1234 + idx)  # 每道题的 seed 不同但确定
fewshot_indices = rng.sample(available_indices, num_fewshot)
```

## 8. 分布式评估

评估也利用多 GPU 加速：

```python
# core_eval.py: evaluate_task()

# 按 rank 交错分配题目 (和数据加载一样的模式)
for idx in range(rank, len(data), world_size):
    is_correct = evaluate_example(idx, model, ...)
    correct[idx] = float(is_correct)

# 汇总所有 rank 的结果
dist.barrier()                              # 等所有 rank 算完
dist.all_reduce(correct, op=ReduceOp.SUM)   # 求和
accuracy = correct.mean()                    # 最终准确率
```

```
1000 道题, 8 个 GPU:
  GPU 0: 题 0, 8, 16, 24, ...  (125 道)
  GPU 1: 题 1, 9, 17, 25, ...  (125 道)
  ...
  GPU 7: 题 7, 15, 23, 31, ... (125 道)
  
  → 8 倍加速
```

In [ ]:
# 模拟分布式评估

num_examples = 20
world_size = 4

# 模拟每道题的正确性 (1 = 对, 0 = 错)
torch.manual_seed(42)
ground_truth = (torch.rand(num_examples) > 0.4).float()  # ~60% 正确率

print(f"{num_examples} 道题, {world_size} 个 GPU")
print()

# 每个 rank 独立评估自己分到的题目
results_per_rank = [torch.zeros(num_examples) for _ in range(world_size)]
for rank in range(world_size):
    assigned = []
    for idx in range(rank, num_examples, world_size):
        results_per_rank[rank][idx] = ground_truth[idx]
        assigned.append(idx)
    print(f"  GPU {rank}: 评估题目 {assigned}")

# all_reduce(SUM) 汇总
combined = sum(results_per_rank)
accuracy = combined.mean().item()
print(f"\n汇总后准确率: {accuracy:.1%}")

## 9. HumanEval — 代码评估的特殊性

HumanEval 和选择题不同, 需要模型**生成代码**并实际运行：

```
1. 给模型一个函数签名 + docstring
2. 模型生成函数体
3. 把生成的代码拼到测试框架中
4. 实际执行, 看测试是否通过

指标: pass@k
  pass@1: 生成 1 次就通过的概率
  pass@10: 生成 10 次中至少 1 次通过的概率
```

安全问题：执行 LLM 生成的代码有风险, nanochat 使用沙箱环境。

## 10. 训练中的评估节奏

nanochat `base_train.py` 的评估策略：

```python
eval_every = 250           # 每 250 步算一次 val BPB
core_metric_every = 2000   # 每 2000 步跑一次 CORE 评估
sample_every = 2000        # 每 2000 步采样看看输出
```

为什么频率不同？

| 评估类型 | 频率 | 原因 |
|---------|------|------|
| Val BPB | 高 (250步) | 快, 只需 forward, 监控训练趋势 |
| CORE | 低 (2000步) | 慢, 每题要多次 forward (选项数×题数) |
| Sample | 低 (2000步) | 慢, 需要自回归生成多个 token |

```
典型的训练监控节奏:

step 0    : val_bpb=4.45
step 250  : val_bpb=3.21
step 500  : val_bpb=2.87
step 750  : val_bpb=2.65
step 1000 : val_bpb=2.51
step 2000 : val_bpb=2.12, CORE=28.5%, sample="The capital of France is Paris..."
step 4000 : val_bpb=1.85, CORE=35.2%, sample="If 2+2=4, then 3+3=6..."
```

## 11. 面试常见问题

### Q1: Loss 低就代表模型好吗？

**答**:
不一定。Loss 衡量的是预测下一个 token 的准确度, 不等于推理能力或知识水平。一个模型可以 loss 很低但只是流畅地胡说八道。需要结合下游任务指标 (MMLU, GSM8K 等) 综合评判。

---

### Q2: 为什么用 BPB 而不是 Perplexity？

**答**:
- Perplexity = exp(loss), 和 loss 是单调关系, 信息量相同
- Perplexity 依赖词表大小, 不同 tokenizer 无法比较
- BPB 归一化到字节级别, 跨模型可比
- 但 Perplexity 在 NLP 领域历史更久, 很多论文还在用

---

### Q3: 基座模型怎么做多选题？

**答**:
把每个选项拼到题目后面, 分别过模型算 loss。选项部分 loss 最低的就是模型的"答案"。本质是在问："哪个选项接在题目后面, 模型觉得最自然？"

---

### Q4: 评估时为什么要用 @torch.no_grad()？

**答**:
评估不需要反向传播, 关掉梯度计算可以:
- 省约 50% 显存 (不存储计算图)
- 加速 forward (跳过梯度相关的记录)

---

### Q5: 训练 loss 和验证 loss 差距大说明什么？

**答**:
- 训练 loss << 验证 loss → **过拟合** (模型在背数据)
- 两者接近且都在下降 → 正常训练
- 训练 loss 下降但验证 loss 上升 → 严重过拟合, 该停了
- 两者都不降 → 学习率太低或模型有 bug

## 12. 总结速查表

| 主题 | 要点 |
|------|------|
| **Loss** | Cross-entropy (nats), 依赖 tokenizer, 无法跨模型比较 |
| **BPB** | total_nats / (ln2 × total_bytes), 跨模型可比 |
| **基座模型评估** | 比较选项 loss, 不需要模型生成 |
| **聊天模型评估** | 让模型生成答案, 判断对错 |
| **MMLU** | 57 学科多选题, 测知识广度 |
| **ARC** | 科学选择题, Easy + Challenge |
| **GSM8K** | 小学数学应用题, 测推理链 |
| **HumanEval** | Python 编程, pass@k |
| **Few-shot** | 给示例引导基座模型理解格式 |
| **分布式评估** | 按 rank 分题 → all_reduce 汇总 |
| **评估频率** | BPB 高频 (250步), CORE 低频 (2000步) |

### nanochat 评估流程全景

```
预训练阶段:
  每 250 步  → evaluate_bpb()     → val BPB (训练质量)
  每 2000 步 → evaluate_task()    → CORE accuracy (知识/推理)
  每 2000 步 → engine.generate()  → 采样输出 (定性观察)

SFT/RL 阶段:
  chat_eval.py → MMLU, ARC, GSM8K, HumanEval
  模型生成回答 → 判断对错 → 报告准确率
```